In [ ]:
from pathlib import Path
import scanpy as sc
import numpy as np
import pandas as pd
from scipy import sparse

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120)

H5AD = Path("data") / "lusc.h5ad"

print("Resolved:", H5AD.resolve())
print("Exists:", H5AD.exists())

adata = sc.read_h5ad(H5AD)
adata


In [ ]:
np.random.seed(0)

if adata.n_obs > 50000:
    idx = np.random.choice(adata.n_obs, 50000, replace=False)
    adata = adata[idx].copy()

print("cells:", adata.n_obs)
print("genes:", adata.n_vars)
print("X sparse:", sparse.issparse(adata.X))


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.raw = adata


In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="cell_ranger")

adata_hvg = adata[:, adata.var["highly_variable"]].copy()

print("HVG shape:", adata_hvg.n_obs, adata_hvg.n_vars)


In [ ]:
sc.tl.pca(adata_hvg, svd_solver="arpack")
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, resolution=0.5)

sc.pl.umap(adata_hvg, color="leiden", legend_loc="on data")


In [ ]:
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"]
adata.obsp["connectivities"] = adata_hvg.obsp["connectivities"]
adata.obsp["distances"] = adata_hvg.obsp["distances"]
adata.uns["neighbors"] = adata_hvg.uns["neighbors"]
adata.obs["leiden"] = adata_hvg.obs["leiden"].astype(str)


In [ ]:
sc.pl.umap(
    adata,
    color=["CD3D","CD8A","CD8B","NKG7","GZMB","PDCD1","LAG3","TOX"],
    use_raw=True
)


In [ ]:
cd8_clusters = ["0","1","2"]   
adata_cd8 = adata[adata.obs["leiden"].isin(cd8_clusters)].copy()

sc.pl.umap(adata_cd8, color="leiden")


In [ ]:
exhaustion_genes = ["PDCD1","LAG3","TIGIT","HAVCR2","CTLA4","TOX"]

sc.tl.score_genes(
    adata_cd8,
    gene_list=exhaustion_genes,
    score_name="exhaustion_score",
    use_raw=True
)

sc.pl.umap(adata_cd8, color="exhaustion_score")


In [ ]:

sc.pp.neighbors(adata_cd8, n_neighbors=15, n_pcs=30)


sc.tl.diffmap(adata_cd8)


In [ ]:
cluster_means = (
    adata_cd8.obs.groupby("leiden")["exhaustion_score"]
    .mean()
    .sort_values()
)

print(cluster_means)

early_cluster = str(cluster_means.index[0])
print("Start cluster:", early_cluster)

root_cell = np.flatnonzero(
    adata_cd8.obs["leiden"].astype(str) == early_cluster
)[0]

adata_cd8.uns["iroot"] = int(root_cell)


In [ ]:
sc.tl.dpt(adata_cd8)

print("Has pseudotime:", "dpt_pseudotime" in adata_cd8.obs.columns)

sc.pl.umap(adata_cd8, color="dpt_pseudotime")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
plt.scatter(
    adata_cd8.obs["dpt_pseudotime"],
    adata_cd8.obs["exhaustion_score"],
    s=4,
    alpha=0.5
)

plt.xlabel("Pseudotime")
plt.ylabel("Exhaustion score")
plt.title("Exhaustion increases along CD8 T cell trajectory")
plt.show()


In [ ]:
adata_cd8.write("cd8_pseudotime_lusc.h5ad")


In [ ]:
genes = ["PDCD1","LAG3","HAVCR2","TOX","GZMB","NKG7"]

sc.pl.matrixplot(
    adata_cd8,
    genes,
    groupby="dpt_pseudotime",
    cmap="viridis",
    standard_scale="var"
)


In [ ]:
sc.tl.rank_genes_groups(adata_cd8, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata_cd8, n_genes=20, sharey=False)


In [ ]:
sc.pl.umap(
    adata_cd8,
    color=["PDCD1","LAG3","HAVCR2","TOX"],
    cmap="viridis"
)


In [ ]:
import pandas as pd
from scipy.stats import spearmanr

genes = ["GZMB","NKG7","PDCD1","LAG3","HAVCR2","TOX"]
pt = adata_cd8.obs["dpt_pseudotime"].values

rows = []
for g in genes:
    if g in adata_cd8.raw.var_names:
        x = adata_cd8.raw[:, g].X
        x = x.toarray().ravel() if hasattr(x, "toarray") else x.ravel()
        r, p = spearmanr(pt, x)
        rows.append((g, r, p))

pd.DataFrame(rows, columns=["gene","spearman_r","p_value"]).sort_values("spearman_r")


In [ ]:
import numpy as np

adata_cd8.obs["pt_bin"] = pd.qcut(adata_cd8.obs["dpt_pseudotime"], q=10, labels=False)
adata_cd8.obs["pt_bin"] = adata_cd8.obs["pt_bin"].astype(str)


In [ ]:
sc.pl.matrixplot(
    adata_cd8,
    ["GZMB","NKG7","PDCD1","LAG3","HAVCR2","TOX"],
    groupby="pt_bin",
    use_raw=True,
    standard_scale="var",
    cmap="viridis"
)


In [ ]:
sc.pl.umap(adata_cd8, color=["leiden","GZMB","PDCD1","TOX"], use_raw=True, legend_loc="on data")


In [ ]:
sc.settings.figdir = Path("figures")
sc.pl.umap(adata_cd8, color="dpt_pseudotime", save="_cd8_dpt.png")
sc.pl.umap(adata_cd8, color=["GZMB","PDCD1","TOX"], use_raw=True, save="_cd8_markers.png")
